<a href="https://colab.research.google.com/github/dee431/Tesla-TSLA-Stock-Price-prediction-using-Facebook-Prophet/blob/main/Tesla_(TSLA)_Stock_Price_prediction_using_Facebook_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing all the necessary Libraries**

In [2]:

#Necessary libraries = Pandas, fbprophet and plotly

#pandas= data Manipulation and analysis
#fbprophet = Forecasting
#plotly= data visualization

In [3]:
pip --version

pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [4]:

pip install pystan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00


In [5]:
import pandas as pd
import plotly.express as px
from prophet import Prophet


In [6]:
#Initializing Plotly
import plotly.io as pio
pio.renderers.default="colab"


# **Importing the Dataset & Exploring it**

In [8]:
df= pd.read_csv("/content/mergecsv.csv")

In [9]:
df #read_csv function from pandas

,Input,label
0,aabid,0
1,aabida,0
2,aachal,0
3,aadesh,0
4,aadil,0
...,...,...
68194,"1"" ) and 8514 = ( select count ( * ) from d...",1
68195,1 ) ) as rwsn where 9946 = 9946 or extract...,1
68196,1'+ ( select wjpc where 5943 = 5943 and make_s...,1
68197,1%' ) ) ( select ( case when ( 4587 = 45...,1


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68199 entries, 0 to 68198
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Input   68181 non-null  object
 1   label   68199 non-null  object
dtypes: object(2)
memory usage: 1.0+ MB


In [11]:

df.describe()

,Input,label
count,68181,68199
unique,62778,4
top,S,1
freq,30,34811


# **Data Visualization using plotly express- Visualizing the historical performance of Tesla**

In [12]:

#Line graph, Area graph , box plot (Analyzing price and volume)


In [14]:
px.histogram(df, x="label", title="Distribution of Labels")

In [16]:
df['Input_Length'] = df['Input'].astype(str).apply(len)
px.box(df, x="label", y="Input_Length", title="Input Length Distribution by Label")

In [18]:
avg_lengths = df.groupby('label')['Input_Length'].mean().reset_index()
px.bar(avg_lengths, x="label", y="Input_Length", title="Average Input Length by Label")

In [20]:
px.box(df, y="Input_Length", title="Box Plot of Input Lengths")

# **Data Preperation**

In [22]:
df

,Input,label,Input_Length
0,aabid,0,5
1,aabida,0,6
2,aachal,0,6
3,aadesh,0,6
4,aadil,0,5
...,...,...,...
68194,"1"" ) and 8514 = ( select count ( * ) from d...",1,131
68195,1 ) ) as rwsn where 9946 = 9946 or extract...,1,152
68196,1'+ ( select wjpc where 5943 = 5943 and make_s...,1,77
68197,1%' ) ) ( select ( case when ( 4587 = 45...,1,296


In [35]:
# Drop rows with missing values in the 'Input' column
clean_df = df.dropna(subset=['Input']).copy()

# Convert label column to string to handle any mixed types
clean_df['label'] = clean_df['label'].astype(str).str.strip()

# Filter out header rows if they got merged as data
clean_df = clean_df[~clean_df['label'].str.lower().isin(['label'])]

print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {clean_df.shape}")

Original shape: (68199, 3)
Cleaned shape: (68176, 3)


In [36]:
from sklearn.preprocessing import LabelEncoder

# Encode the categorical text labels to integers
le = LabelEncoder()
clean_df['label_encoded'] = le.fit_transform(clean_df['label'])

print("Classes mapping:")
for index, class_name in enumerate(le.classes_):
    print(f"{class_name} -> {index}")

display(clean_df.head())

Classes mapping:
0 -> 0
1 -> 1


,Input,label,Input_Length,label_encoded
0,aabid,0,5,0
1,aabida,0,6,0
2,aachal,0,6,0
3,aadesh,0,6,0
4,aadil,0,5,0


# **Building a Text Classification Model**

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    clean_df['Input'].astype(str),
    clean_df['label_encoded'],
    test_size=0.2,
    random_state=42,
    stratify=clean_df['label_encoded']
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 54540
Testing samples: 13636


# **Feature Extraction & Model Training**

In [38]:
# Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Model trained successfully!")

Model trained successfully!


In [39]:
# Evaluate the model
y_pred = model.predict(X_test_vec)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      6674
           1       0.99      0.90      0.94      6962

    accuracy                           0.94     13636
   macro avg       0.94      0.94      0.94     13636
weighted avg       0.95      0.94      0.94     13636

Overall Accuracy: 0.9413


In [40]:
import plotly.figure_factory as ff
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
x_labels = list(le.classes_)
y_labels = list(le.classes_)

# Create annotated heatmap
fig = ff.create_annotated_heatmap(cm, x=x_labels, y=y_labels, colorscale='Viridis')
fig.update_layout(title_text='Confusion Matrix of the Text Classifier', xaxis_title='Predicted Label', yaxis_title='True Label')
fig.show()

In [42]:
# This plot cannot be rendered because the Prophet forecasting pipeline was replaced with a text classifier.

In [45]:
from google.colab import files
# Save the processed text dataset to CSV
clean_df.to_csv("cleaned_text_data.csv", index=False)
# Download the file to your computer
files.download("cleaned_text_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>